# Nepali (Model L) — corpus and tokenizer, self-contained

Collects the Nepali corpus and trains its tokenizer from scratch. **Nothing to
clone and no GitHub token needed** — the pipeline source is embedded in this
notebook and written to disk before use.

Run this while the Hindi (Model H) work proceeds elsewhere: the two models are
independent by construction — separate corpora, tokenizers, vocabularies — so
there is nothing to synchronise between them.

## Before running: two settings in the right-hand sidebar

1. **Settings → Internet → On** — off by default on Kaggle, and the single
   most common cause of an unexplained failure below.
2. **Settings → Accelerator → None** — this stage is CPU and I/O bound. A GPU
   does nothing for it and spends quota Phase 2 needs.

Then **Run All**. Expect 1–3 hours.

## What it produces

```
/kaggle/working/nepali/
  data/processed/ne_{train,validation,test}.jsonl   the corpus
  tokenizer/ne_*.model  ne_*.vocab                  trained from scratch
  reports/*.json                                    statistics for the report
```

`/kaggle/working` persists between sessions **of this same notebook**, so if a
session ends you can re-run and the build resumes from its checkpoint.


## 1 · Environment


In [ ]:
import os, sys, pathlib, shutil, subprocess

REPO = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle').exists() else pathlib.Path.cwd()
REPO.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO))

# Keep the dataset cache off the 20 GB persistent quota, which has to hold the
# corpus and checkpoints. /tmp is wiped between sessions, so a resumed run
# re-downloads - replaceable bandwidth in exchange for non-replaceable quota.
os.environ['HF_HOME'] = '/tmp/hf_cache'

print('work dir :', REPO)
print('cpus     :', os.cpu_count())
with open('/proc/meminfo') as fh:
    print(f'memory   : {int(fh.readline().split()[1]) / 1024**2:.1f} GB')
print(f'disk free: {shutil.disk_usage(REPO).free / 1e9:.1f} GB')

probe = subprocess.run(['curl', '-sS', '-m', '10', '-o', '/dev/null',
                        '-w', '%{http_code}', 'https://huggingface.co'],
                       capture_output=True, text=True)
online = probe.stdout.strip() == '200'
print('internet :', 'ON' if online else 'OFF  <-- enable it in Settings -> Internet')
assert online, 'Enable Internet in the notebook settings panel, then re-run.'


## 2 · Dependencies

No PyTorch: Phase 1 is data and tokenizers only. `sentencepiece` *trains* a
vocabulary from scratch and `datasets` is purely a download client — neither is
a pretrained model, which the assignment forbids.


In [ ]:
%pip install -q sentencepiece datasets pyyaml beautifulsoup4 lxml regex 2>&1 | tail -2
import sentencepiece, datasets
print('sentencepiece', sentencepiece.__version__)
print('datasets     ', datasets.__version__)


## 3 · Write the pipeline modules

The next three cells unpack the pipeline to disk. Each carries its files as a
compressed blob rather than inline source: ~100 KB of Python with Devanagari
literals and nested docstrings would be unreadable pasted into cells, and every
quote would be a chance to corrupt it.

What gets written:

| Module | Responsibility |
|---|---|
| `normalize.py` | Devanagari-aware Unicode normalization |
| `language_id.py` | **Hindi vs Nepali discrimination** — see below |
| `clean.py` | Quality filters + MinHash/LSH deduplication |
| `splits.py` | Deterministic content-hash train/val/test assignment |
| `sources.py` | Hub streaming, scraping, and local/OCR sources |
| `checkpoint.py` | Resume state, so a lost session costs minutes not hours |
| `train_tokenizer.py` | SentencePiece training and the vocabulary sweep |

### Why language identification is the interesting part

Nepali is written in Devanagari — the **same script as Hindi** — so a script
check cannot separate them, and as closely related Indo-Aryan languages they
share much vocabulary. Since Hindi is far more abundant online, Hindi text
leaking into this corpus is the likeliest way to break the independence the
project requires.

The separation therefore rests on grammar, using the highest-frequency words in
any document:

| | Hindi | Nepali |
|---|---|---|
| copula | है / हैं / था | छ / छन् / हो / थियो |
| dative | को | लाई |
| ergative | ने | ले |
| past participle | किया / गया | गरेको / भएको |


In [ ]:
# Pipeline modules
import base64, gzip, json, pathlib

_BLOB = (
    "..."
)

_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode('utf-8'))
for _rel, _content in _files.items():
    _path = REPO / _rel
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print(f'wrote {len(_files)} file(s)')
for _rel in sorted(_files):
    print('  ', _rel)


In [ ]:
# Pipeline scripts
import base64, gzip, json, pathlib

_BLOB = (
    "H4sIAMC8g2oC/+09DW/bRpZ/ZdaFEdIr0bKTFj2lKpAm6bW4JimS7BaHOGApcWSxoUgdh4rj8/l++72P+SQpS053D0WxDhBb5Myb"
    "N2/evO8Z3RypRVNsWnU63xZlni7qZrNVyeb6aCqOvvjL6VY1p/OiOpXVR7G5bld19fCiujjCf99hB1FXUpRZdbnNLuUDJbj/FH6X"
    "pVy0I1HVzTori/+WI7EoZVaNRC7z7aYsFlkLzxT81SYAsnq7kuJqVZdSqBZgiULBH43M1kV1KcYirxfbtaxaJbJGimVRtrKRuciU"
    "aFfyGh42xUcpsioXV03RthIgQvesuFy1oqjaGpsVDY+H3SXAVLXYyOyDWMt13VzjiEWVy42E/6pW1Es9G6EAfQR9UWXiy8nkxbit"
    "P8gKGl9K1YpKfpQN/C9zwkU02ZVo5ScAUIm8UB9odn9TMKfpFP8U8MOkFDtoL8bjRV0ti0sxl9UlUO+UP6rTTbGRZVHJ5Dpbl+Li"
    "gqHhz3is6m2zkKrXRz/nLuPxpqlp+ipbb0qZG4y+ED83db5dtAXg1Wwr0WbNpWyR+P+LcxZIzAo/0uTV9H4TyRSMJ5W810x6nXZM"
    "ZbktS8OVOJ9lU69Fmi637baRaSqK9aZuWljBqm4znKDCVuZpc7nJGiXtA3Wt7N9tsZYa3iZrV2UxN8B+ho8I5fXzn1+lr1+9eitm"
    "9CyCcQGnNI2TRqq6/CijOIEBkHXfnb0HrrwGykDDpKiUbNpoMkI+jyycOLZTKNdZkmdtlixWcvFhUwMfm+Gf2icvsgpYqxmJH4kd"
    "37S0r1pYQtxh6W+qrsq0rXsgcTNaaPhBNk+JzCPxTO81/bjb1W5p0/075rg3xAQvzVtACRgo1YvUBWLWWIMoYDundo+PRFlnebrO"
    "qmIJc+r1xU1su77BT0+UKi4rHJM+mpnQh18ahO4BAQHTqoS5NKUPlhD0DGlYqLZYKK/Tti1K7ETsrJsTloaTU37X61LWl5ewb1L6"
    "ZHoq2W43qX41IpnFa3VB/3K5FMSVKbCnimIx/tYyavISN8UmW0i9B5nxf8Z3ILHWQLV8jAhhD6ZnYvYGbVls2AC7WoBPdDsC0US5"
    "5O0MG2WW4qKk6UgscVlbXKVFCftyZju/zq6euQ4/yHLzvWka+wMmWZ6nBqPo4siIhoujkWjkf20LEOezt80Wmfd6I2e4mUZiBfBm"
    "MDlNY/GfT178hLgEeufiaM9ImtkOGuoNtRWG92jEvQMYJj9C9bbMtmULkIyEPXLTMAILOQR1heEdofkq0n3+B0VavHfYslgX7RhW"
    "iKZGcwGJ4HB4CVSyg78CNdUUOQwqG00Rq1UFQbp7PCugcWTWDqwIcXD31mLhHgXouMddvJAgRstoFcvDPDaqVrX1RsHSA+ZgGYBU"
    "zO3QB2O+WEGTMdKAhujjvoRdPYT9o2TSR/45iAlgdim2CqwRMDNgIWE+LU/GmwMbIZnA4bMFbA4x3+b4fC6BnaVrj5JTyE8gf9RB"
    "k4MpNRJlyhjtkOthRjifpJPJxOfDS9BOSmgBhMjJ5mNWwh/O0roHOzgFNc6Lpk9T2mGHMsQz2J+LFi0y3OiAJ4wo3AhKRLr/VHwD"
    "Gw82NtA+hXG/PfVaxYkwEk84VF9VIOPLbD4SrE7bFVh9WSvW9RZIkIt/r+tL2KDPyJoE+xCMRbVtPsInNu7+/iL5HJaz1DFrdNB+"
    "Of+SVq1HoDcZmroeSQSBFS/FB7lp3QImYoIWaDYvAXvXGhb8MybB6xCgnpG1iJIOVkumYHOE7zW6r3kFrdDzrZlKdJlHFEsS7vfY"
    "Al08Yce1n4PoM1nKFi39ax4dd4aHLDoXBJu5ZuMLc1CAWbtYddFtQM03lcHa1+hOz69B4rGGh0EClf4azXAkGLshZFz3lF+g2xE2"
    "aPZwIHyh9cts0GSJsKU2bUYEI9GT073NWNCfW727OHLK971Zmnal/Bb0wL3W5PIbGK353nghII8U7mRo5QzrUwaNY+rXFijKL7Jl"
    "AnsqMu3A/QRbaba8OPIdk/TG4H5rRRwDApt8WUfataXVZ8ojzY9B7hyreGQmMjsmpdujR4pDAoIjSzXbBVrpl7F1BH35tWPWQRs7"
    "dbJb7+hl33vk/QJ22xjdKrkGmdC1QMAyrcjDHh/wwwCtK4BY7HADIs8LiAxNYsOX7GJA747TEQU+ie03c0Q9ObG0JyiwXkhZDTjT"
    "DgFADhyEyPMPIg8E+xQeAGWdAADR9Qv6+NgVVRtJXQIPhneYNkMHWe4naA765zgXskKBDaKGWkcqngpmtVJWEUEHJsTPF0fJbyCY"
    "IlptapwWYBC9J3bFWAbjElvU0JQnQ4+Nf3BN8ANa+Aqlbu+RgvVtxUuSxiU4F3oRE1hQwBhMqFQPawHb/cRGT8r2oRkufGhG7Dy9"
    "e1DTGM3E1FifseHut4ERldeSgZEwF9cSDCJVD1hmCmgONFugBbdq6u3lSmQGZM9e2yoUDOA7S2365bR10Jyi0Ak8SQgRsE8L0Ms5"
    "D2bhoaGBQ65lBpYFvJafAH55DeKiRf+1cNEUMxUXUEn0rgGkUo3PDNVGFBLxhClLpm6K60TPYyR52JAojLRmuPDeAz11CjLg1So0"
    "qd4yFUE+0qro+BDqsx7tUIaCsXWcnPNAeopxoIzxByT2TYDpdHSLLI/PfQz56dBkPYBxKP48jU5a4yBxt0P+OVhaEmtUgqewG6NQ"
    "zp8C2TxT1XLwmmM5KHC68Z0oBOq0S2yXjsbW5g8ZKxpcwqZUFO9aUGc1iUuwdKvHYIGWHPsbsoNgjbXuC1CKHXQzMMrlJnKiEWNT"
    "MDkvUjUk2I2CHdKabFSBPYk4zcT3WalkjwBobe6dP4nbmW2DDyL5CQQm7Oh0Hwp6QA3Fk1nTkI8VR+Mo5J1usmvsoHUDhl79pt6s"
    "MDQRvty9/3gPvjadyRh16zIFFr8kB/84H+H2I8GuJOwP/ITuwsjtXNpCvb1o55FoWEr8VZzxXjTPZeVtUH6IsHsPWW7TOPwuHCoO"
    "P34hXktQ0ItC2yr1tt1sOXqvxFXRrrruhPalsw3G8NHNEU+6EPVGFB+KEpXsusjHFHwTCxDpwK7gWa0RBtpEyoTwYZyMTfEyw3Bk"
    "CNJDYMFu5GOHAoJo6w1mE9pVDdL2qt6CFZ/XW1DyY9YGAHjdBQrLkRnz84o6egbJosyKtaihX3NVKJl0hGet8xxkio6E4WpSLbxg"
    "9BZYZp0111qzem7/SNzcxgkQZR1uGvODJiZ6GR2ZhstsLev0xqFwm1AUuueOs2e2zUoA1gtYRxuKyhnk435f3PHc/S8z224AXW8D"
    "XWUN2onRcCPeSwI2xDRwrg39YKMwW8w1F4pVhk8fi6GpOYhdp1eJbCWhJ/GEFBjTJstE1WvpJbqugEWBpUEyjDn+5YW7hn4oq8BL"
    "jkthKeI2IZNraOft2IH4gzbC3VTVWoQJ15gtm2NcxZAMMejg5yFjNYT50W4BaTP02MmZiW6GRB8Kn4ujqXCiCC1jlD7uKcu5i6NG"
    "/kYkcW/Mk9EQaOyWaqHog+JnZpj0qm7ybgt61gF62yEu8G+oGHbrEZ8m5FhwRxowCoAMLGB3lTRRcy8Ti5Yr5j4/WVPEQ5Kn5adL"
    "eODpsJ5gGcVoUjsQD4s22gVFjwdOaKDA77ZVUMOTqUJbsar97bUEmZqzkfKY4zWsGKVaDRotnm3iabiZODMeuvxY1ICzLLONIpuB"
    "2+rPqUKGBwYwYJqWGmHqMFnXsKB1VSxMHMZTgBaQ94wbsXGb6hB3aOZ46HPQr29zcozR+HwSFhC0WuoaRKTwGiP/pwLXhwJQHc7T"
    "iSXZKPSfUFRhWsIF48m2WZZbtSJF52vnZdGoNghN+USGdZtZfuZsHG3jTkOKZnYb0s7uNDS7GBoTr4U9zNt4CL5di/4o/pJ0OtH2"
    "Hu7E0qDTKVz23oJ7tPE1MzQNl6rbur+bNFPpPehtv97kO+xL+rzD6X8VUZeLsWqCWXzA3Ec+izRbGMr4gspFIsh485KzoX/kh87Y"
    "kJoZ6zjGig8myjQwxa35rKO0XVp2hBWDsPqF7DAVDfQLVBNaf95GSrN97Nn2BkbjzGbdQEiFie8Bw4RiOAPqyQu/UOhlZoMwQ/b7"
    "hwI0RK0Kqn6Y6Xlyb/t4oB9GNjAmnuuhTNfe84G+dYVuFOWZwLlbz/NM2CgVpl941KnYYYztGChhhogsqHi4O7CEGULMZmJ8xoxh"
    "epGitfZwb5RhmBQtGaZdksJGatFkTtNoYJoDWMZd52dAn9r4Koo2F2x1JRiRYZ0ES44GRunIUL3PIgd42LCmbmCP7LBFdEoBNCt6"
    "qkNMW3KUUUddE/4jYkPM4nzXIuodSr8igjcSu7p7z3Th2TDJPdGY5Xl0X3CDlNJTnaGdj3IdDMFhevk6AMwLjOYiArv514/y+WFR"
    "ZGQf2LezHVG7uw3Bp93onLE3ojAgED+mvPuG88So+o070fXm491D90yafowjaA4NP+wgeM8CQnr0hTBoqgFxTbQK++8gmOY/Mm+i"
    "HTMbNqwSrTqieEe3e6uRw0NBznTzw5zZR+uNmQyx6i3znb4lBV57GHqu5Z7ITi8W213UnjEojo07gMUNxtidicmO9WrY+okGII09"
    "wzMWp2CvfIrO5PjfRuIg8ybQugtcK5iwiG7OJhNxEuzE02AjJmfL22N08vkzELi7q0mhhCby5yw1zmum13ZGeUoY+vg49nGD5/Dm"
    "OJksafVP1T2XuxPj28ULu0EytQb2qV6P0QAPxHtQ7DEdrs+IWGF/fEN+WuDw/yGv53XW5D9iCUyz3XQF6BeCyi10aFF+KsidVFjL"
    "k4PTOpfEeCghYb0UZ5xWVGJRHSpO7ilG9sayLo7sZDAWuU8SJOK1HGPJL1nl1r/mYiZS8cngyh7OB3E31E1VEWcPJ740GHZ9HBVM"
    "W+eHf5bDYnLFuvCzK3gzFfhLQZFo4Ft5DW3yhisGZ+LG96FdocTU+TVDDXTlwPSOmgK/27oG/gua6ydhM1tkMTWJleB1N4oBzfSj"
    "oFmov7FV+CRobKIofliOnwTNmM8ImPHegvemPnQK9PRS/N6Evbw/RuNMDt9rYdP6tyGGpiB0Kt7dfJiKj+SXfRjBHzptbsLfZAxj"
    "gPniCHgKu9zqALtc2BR7SPGQwwid4EmIic/1RDD/QYcejhd5ku6zbnhrHWxbUBy5QpFTB8QwK0XmUXCaB/G+/sFUbPfwcXwYEh0Q"
    "3sPYKwgKYn9jXYqhi6GM42YqdDDqB7qPSlioZMLGETRTD9ZePLMhd9JyOrtpxZud1rsH+P7Be9/Y3QWJRKFwyhgeyA1ppngXcOyi"
    "gY98demaYMBZblKE4nKRIQa/UGTKjG6H8mj77kFbt1mpHrx/94CCVeGEgBG2WZkuGy6ao3otrzMunnGJUuJVwARlD3dDgLZvp0zL"
    "oPiCmgoHRyjQ4HJKdQHHZDFxfC3SxdpI0SmY7ucTMGoccTqYukR4dwrfiElyPulHlIc1qEWQsELPay7L+orin4iB8LBKxHOq1uGc"
    "IRYBbuTpq6evhTne0DXtLo4KTiSa/TYSVOOab6ka+6rSKWUTVvHLgy8begWaOvHB6nnbrN/I1JhUoSx5F+T43g/k93rpnPHZmTJZ"
    "Y8oTmwyxGYl5C0bzM73BoA9sEQgyHL27Y/+8BSuKw7VUSyMzkA3K1j+B5o16O8LldOyU4nfTr3bsjqZug0wqGDkM3Aoox0adlMC3"
    "4mwnrZ6aAJLIFk2tFNZxWZ1HIkmsqTQGNp4ZzdOK1qY5FV9NvHIVLCNCw2xZVIVCLYt2J1hPheYIZ9kl4ieZfSQXvdUp5nX2AauE"
    "qazIAGzY0DO6FxbZVuXKihKSmYv5mfQz+tdZnncM2wFPfHpHBcgOuuFrHMmZqBEiaHBwRyC0zTjhulkYPCWjKE059JKmWEabpjb6"
    "Ap4B+FZvrhWwxHMw2iMuswVoQH576NGrI6O42iHnHt+Sz4xEkR+zcovWf+cQpIWH1VyLckt1AEjjj/Uim29LtG/VFUhyOhr4Bo87"
    "KjoaeJaIk5M3+ObkREQ1najJypH49dfxmDr8+isatNprBxSKHIe34ynNfkQCfzB4yes4hzUUWXlZg2perYHzdBmaQDMOaxVWsszH"
    "9bYVMDeEjmIcoWF4ccQxVIlim+YzR3N3fi2WsmkL2Nkw0nb+G72tsURuW32oQKaNyUVaSGhC7HOO0/y+gKnZ8x4w3zGweavZGhyp"
    "Shtx24bTk3UVHBCxJ0cfIrDnvBLQjgEZfEYGBZ78dSvHy6ws59niA3uJfu0e21CbQi58ETHSxYJLFPuyWhQolSkEx1Rn7OQnOriD"
    "r8pSaJpZWiLxDjgG2ufGQ0+Cava4N9T7HcvEIeD3Vkn+ezynw3l/zoOXuw8JckaOtBwurOp1srQO+2k74615q5m2brTxzhsrRaLq"
    "wnzzAomduv2c0n6Gl7E3tFvgzoKb4e245nAkt0MAXHTzr8OOf4rDjv+gs4j7D1WSxMHy28HTNfb0jz7E4th3TOfpT50aEhrUwWd8"
    "OhLoM476sBKnkgbWQ3a32LpCGuABlmFVFZ8F4d1QgUnhirf0UyTlg4MP7KFiAL1wx8nNh/5xvb9bVexVh9GBQ2QOaXXf3iWjNSDy"
    "3+e8qKNSx6TYOx4FpMY4js+NPAoo+QITymDCb6sCfJs1n2mYb/j0zBAWlmO6ZuFnHrAiLhjxSo9Cky48j/knPXRFaUmrrBbLS7+h"
    "fRGezxo6b7XnrJY7htVRTftPYlmlpW8F+SOexHLT2d0zaPO7znBprc2rencJLjUNa28/YlzkkL7O/A8B8JFc1mAYYogcPiMLPQ5r"
    "czBzbfoMFeMP+muAzotC0WGbG9P5NhGoTnbdNmKKzjwP+gffBMdYjo6v0Bly3PCN9AQpphidOKUkD94sosu8jb/FV9CoBdVQsu1P"
    "B3hIaWRrLujVXrKW9CgDMHrRNR0jQ7ERn8CfkRzQnegUVuziEUhG/coj3hDhPIXB5Qk3ZpxbOu203rTXj3FCdDrKebFdj+Voz1G1"
    "AWdRBHXkWOSgUXbFX+zesT1ryhfpA9m1GA8LZBLsA/fWbh0NBHWLgUHahp8MwHBv3fYjuxpDIWWrprRC7zCn8x4gvuscmmTxczYV"
    "xgsa/45jQ6bUlWB5a8l++qyHu+cAsO1vp2BOkO/u5eYd9grXlIIOuNd4VRgRELQ13m1EIPQwOlpGDgg/CisL76x+7REdNcmgc9OJ"
    "yTohAwaD+/C+k2c0G4f21kx/6rTpHzgK33tUnhk/y3/v0XNG/3ffZ5/08UQqfJ+F5xf1moRtYJ+NevHcHWtkY4E210Gm3F1ZTwfP"
    "3KdQcnQ4YP9e2ffFEVhedYNVC5XuNd1ZXG5D6RQ2/loxGWfH46+U+P7Jjz89f6ajuLsT+t45qLLtbtiRex4IA++5xvZ9J9m84/jC"
    "/tqKoanYwNLsOHm4xOAS/HEuKbKULqFR8ggzGPZsI75d7pz1Z0/VohEmewdhA44mYWQSJ/Ydo60DYq6Vfd85Uzk8WsBhc06A90MZ"
    "UcBt4TE66jRcbNhjr5e1+FhQyoUFsdXKjynD5hw1OlvrLHgwWLFwQwVVagO8se+kHW9C9AllQ/k9zR25iELmGK6nwqn2F1w/7Sy3"
    "fnrHYvfPapBaAYsmZXIjiKGjJKEGxlZaPlGQE08apT4ywxjGOwFrrTw02X3HU/5GFp/yqExSa8BT7xwb2pls9qJxFHM3+eabbmZu"
    "T6GGK1oodWmDz9G9hrwFdK0EJvqGG2AatdlyfcYgw32Hm8MaWoOR9iiLhdnm4puZOJPjR2TcRvMYjL25zCkDMVRmhg4juN4S83PY"
    "9dHk2EQ2zr98IX5eZWBcnpsK0awVORsUs4dfP3qMzbCEGpYKU5c7BqBUAOC7BjkDq+nFEDDUArrl4bGbViK+tzPMSnS5OZmVlVfZ"
    "tdoxgpeRKPHQOBDMigW0VdDWx1sfAQElrJaET4yCZ/Mk3QHiO5bNUY5rVAYwQ0XvBPCUVmbwqBnvkbxYYyug7WjnUunaQWzH9wN1"
    "7ggKB7ern1I2G3tNkkfD7e0igLSHlcqAbNx+8rB7ks37fBsPGcrnU7NdTb7mc+xkApHaMEongh35BzN2G3VO/Mw88eQ1cEJt5gtG"
    "r4XNYQMuYBDjQF1ju98kFNeBqu31Dt6GHdlbzIvLAkzanonvvQy7mQL/zEoYCof0IOxo18FByrw/tqSkexjx7ERvzOJ3RDcqzpHV"
    "nFQ44K2058h1XriVig87bufV5HWzHR73BB5FMOIoDO6MyLofsutds75tv49AHEWwh4tdDdO3u0xmDNWYqb17YLYYD2griHb7BK7a"
    "y2pnHRIc3MsPp35Q5LOdXmnSXf5OtjmwiHJ4O/Hq3XcBgrfhu3ks3MREMmzIQUf2MKNrSzhdc/dcX+pjKmlcpjdoT2ucmrfXnVFM"
    "BjjESD80bZmBzNPZ2bklOJe5/8FqRy1n2ZtYoYutXPJbeiGrra399KM/4fC8eASN/3TltGHxameJqIrVLWeIbLg82NTP2QdFr3oB"
    "qNrVZO6Dqkw27TI6WT6v67Ljvrhyy0NNz6DUkRf7jiJHF7a0DoxheAx9d8sbB2XL38NkzVRPjuWtIXtPooYwrDE2DeQPurn6np9T"
    "LNXzIVorYhDg33Q5BhpEUx/gQ+kDMTbT8O1svTKOqUHq+NgrD9Sw+k7uINSndIIDfF2e2NRhtvQx696ptAvBA0AhYvtAvYHdV0p9"
    "GypVReq5ng3OVVFzhse1mDvnS+whME+vlf9UOMjoA9A+45JY1ElBaHhQKzEq3p7TupF5zBsJ8Qar0oS2GzywD/hvYJWKEvx2Cgtu"
    "19EZh+U/4fBWvOpAuPyEtX+2Z/1hV0Hsa2wzxkbCgJ/SNsA5HufCyYLRACYcwjZt4juKCjtlhHfs/39uRVvvzrD9FW1PqV6U0hjm"
    "ZjSqECYW5gQCepOmHg3cpk2jeRID9uDg2Xvuf1ld8+2YeMkJ8A66LK2+0v+iutNGwGJH9jfP/BJb/iKATN9fc6JvF8uWrWyCEqyT"
    "kbhaFYuVSWvMYTyWNfBYXyCkCxqppBfNUkwg4UHBds+9cXihW6EuKiayWPBBH/I363ozReey2VbKFagFoSeKoXNyyRLzomIdwH18"
    "0vNRdLxxqLW3wyjZ+kWrVH2GpaFsM6y3WIW1KNDZRZ/5Upl8uVrBexR9IrvMsHCBnvJ1/nS7Gq8Z+sBYYkh6bAxM8uuvOCXggBqW"
    "sCnooLAuaxRFTlWzmViWgOG2qNqzr07x18Nz/CKFdba5qLKmya7NehQ8R66pMRGFB4omxum4hq+6AVsEi1KR2Bktu2G4i2olG7qO"
    "1/AHEOMa6DBHeURwvII88McPqbXr7ZMDa+3uDW1PjZ1H9j9ZKZ0epdqugRwZbKmNfcaE3HOv/88//fg2ffnkxfM3I66561ap7S15"
    "++7Vm/THZyPxXP+2bgf7Xc2/ytf+eOVrXnw3gL6/RonjBsTke0dxe25vAdsTFIN8R5yTgJkyEpBlniCR98++QP9LjvYN3wr+2tOZ"
    "dHZzjVuW66bsVaTBl7jogyJaN3zO3dXLosrdtouCYMlU8MKZlcezgA3tDnwe7Ief6oWtueqpT15V5KPMq7/SUv01GkV+8tLZR1Px"
    "C4anq9pAQCuA7u0GYVy1iY+ArilwFR5ggNYYKuGwRDCvOLks63kUlsucJNqB7tRrOJB7SjZe1gNT1hdZ3QTjd4pgBsq7vToY89U+"
    "cik5ibKuFRbjLIACYMfBQMWywIhQqKspXBdVdcVlljFjlASMgae23fTwco5rc88LmEQbqoUGvaHadI0q6KCCPLZEtY3nanLYbKqX"
    "v6cGD1VOorKlTOmqT6/6DpRcllPNQAR+S43RcthO23Y5/tqt5++txvsH1c+Fdka3fu7/905xF1zXFWV8nbOze5uufDikMm5PVEOr"
    "cJRhx8pekIkWrD4kZkv9HGpdyoQhSG0L8A2wQacwOeBK5uomoTvNORCFt/DYncb2MHivy2XBX8aFRwe/+vLLh18ZhxpN1VVWmm+U"
    "gD1GV4xy+onvbNP502qTaHggTTxEvhHnJyfwlK6P0I0enrtIoq5nwnjaO5C5I2GKmm5uO2cFUbp4dta0m3Y4qFSQT/wNXPKpZaAD"
    "tav4r5flf+O+pm3NRYAjusMK8/vmOKAD20lED11MdEhlkqutnqFXbvWxucN8MpAQ6jwHofF1SnGd8Pkc+AHTHlRiBitW5WQvcKGZ"
    "zduzZTnHaAeurCtBs7U7wEvmzjBOYnhE6NCUCh5n2AU3GcVs3ndvtniOh77cbWQgcjGjmfN+AuP5FAxnTzWsYHiTEUaPrXe3Lsbj"
    "FId/pHrM5oU++7YW5MNhjYb2k69qDgsocvyAME4pWwaie+neGTP+xO0/EtQy4lNpVPKlrabYWPudybrFdVc7dlZYX8oEg3YZyq22"
    "bjNw2Za39F4jg2hPn3RrNuzaD5RrMPeYm8+AfTJF7IOojlhazOj/OKiIcBdl7E4Fea2rlq7d6JVFBCF99xVt3WbGkp1qcvYaOCpS"
    "4sF+6jV0lMSG7tMgxKA8yQcLYsp90YB/3f0kmfQgdUKx4bj3gsTvCZShlaMJQHJ8WCy9Dzvh3Xkvx62/3H0pQmceWfKEYOFleo9L"
    "nG8TcpQ6FxmWdG0VcCPe9wsmYIWZNx4u7jcFu4C+AMQM3a3SQ+7DRLjOw+LXWMwoqHBAh1zXFjE69Cni/UDXDe/uqJnsvf4SBcKz"
    "oofxHVVhNuDLV2gxt/Mh7xfFd3RBgUGaT8QTXMrB6FPybhyg+Nnk/NHJyXmwdbUO1+flMaWIWB/2fQz9U/sOxfB7F/huioGbv7zT"
    "/ZZF/bu+/K9kMPQcqBUc4Ng4PMZgdauesMmm80V6fA+5l2AnvT8xV+lijNpBwISBwcU05q9CYclmi1hVgulZqey9Si5KOiOnZjIa"
    "+mKScYDyHzxv66oRTAfcSp5xGzT3q/qmnqEZpmLNyQVzX0/Zzb8G38kyDcjVwc4tHMtG9zlsOPRlL9OBxRnqteYaqYDPvp3t62x5"
    "wRvPPgtampiKvWYkrZcpA+6Ne3r499/Yb2S5T4pZ3197eH75eT/hcHBe+W3ni4C9a2b8Se++5OMJ7DQO9erMqAfAY4WdAKqB754J"
    "rtoxSxMKNrpzvkfw7nVlF0cvnr/lm/qYiu9CjnrPKwTA3vzw6vVbvP3gxjJIFxhnRCML6E6eeY8O8yQOg3D29NQQKoffaNNZMzrA"
    "pG+14TyQ+F4X2bpLZkB8F/gNE/oLqfiKm/59NvaLfMN7bYR/zzEaIr9tVVssrzt5Kd2L5zd0rc29Uq3hPvjH51lv/w8BTYFhL30A"
    "AA=="
)

_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode('utf-8'))
for _rel, _content in _files.items():
    _path = REPO / _rel
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print(f'wrote {len(_files)} file(s)')
for _rel in sorted(_files):
    print('  ', _rel)


In [ ]:
# Nepali configuration
import base64, gzip, json, pathlib

_BLOB = (
    "H4sIAMC8g2oC/7VaYW8bNxL9K0SMwE3glWTZThwFLeAm6bVAcg2aHO5DU8jULiWx2iW3S65lXdH89nszJHdXspO212tg2BLJHQ5n"
    "3ryZ4ebXB0bVstTj3JqlXrlxrWtVaqNGO1mVD2biwZH4VptCiy/e2EKV4ttH4u1aOiVORVoqwrNtI722ZvTBHOFHvN9aUTd2qUvl"
    "ZuLayaouVXEttBN+rcTCtqZQhWhaI1qHD96KG9Xo5Y6nO9nSFCQNkoo2VzzXqNo2XvCWyj0X18u2LK+Fl81KeSc+Xkwmb4RvpDba"
    "rCB3o4wbiXeqVLkXW+3XJPA6y6J2IjxujXB5o2vvxotWl8U8t03dulG9w4k+mFKaVStXaiaM6r/Njaww9E+24QdTkYlm4vUHE0TN"
    "RKFupJEr2WgSUku/drMPRohGbueFbkgam7+QXo4xSHPQK1cONrm7opuidXwy/R/V7K3rRmmJ89K7velgPEeTpV3tz9EAqXkkXpRK"
    "BuutYeK1LQvY2SklykqOSJNRTitGvE41LxgAYmkbsV1LL5TM16KwysF0eRTFx77PLEIcidd2qxpsJk1A27FLRhVbtYCovK2UgXNl"
    "owT0aTyWw2MSkIEbTgJMSFKhXS6bIuiuKgFtAn77k9BHu4WsrdwJKaoWypYEngaiaQO7jMNBpKtkWWIyAoIGK23mOdZCzdPpJI1s"
    "bVPQyAUPyNu0BIDEvyDsm0bmFCa0yeNSeRzEPaaTe+zovFgo8bIzzijqTjbw6tYLHK3RCqqLpdRNkCgrhJInedrkuoCZZClemVWp"
    "3Vp8sWhgGkEwdSeQka+NzjGPbSv36EQ4i70Rkvgx1sfzKi/WerVWzXOOtwT2jIVrvxP5WuUbKOPxwYmVMi1Ctdx1u3b+6owVHD9n"
    "ipiJyejpJNkIgCxlrmh5Pz2ZTNN80dYlVPZqTnzQLznrJHhbz82qkVU/Oe0mC73S/r4Jt6sWtuxnTidDr83BSzMxjU570WjPZiOE"
    "B1yOxHc+wCXwWe+0CHJmmuA/NrOM40FisqCB1WHwWkIPZreKsZwwSz6Ha5ayEZUFMuUCrCnJ24bZMUO86YTTSm6gDMJtA2cBDD76"
    "NuA2uFJvQKsKOCPsY8GiwXKewoaqVvhlchV1tMYxjfqReE8rkvvBmhR/DoLJ/bB8BQW/yG3dlhIwyyk/YGgDbD8C1UF8xCpHONkX"
    "QYDJjlMCu3WsqotAuyLYaT6YCCrMwO8tq4kc0tZzdQuBh4NGyaYfo28EpnlHBAMcaoOctp7XiIrWcxpD2D57wpS1BpeUau7AqQjl"
    "wI/vgEkvGDuOQUEWpDySgX2QUoCekbjy9+eiE3H6kLzqvC5Lkvbx4k2cEmtVFsK2/iQ4lpZx4uRNwOYLfISWGLmFGYJfutwaEyUn"
    "NQtmbGAu9rsjpJCpr1NGvgbRlHZLyVq8DtxHW2fYWiwjRbk9Qj7BE7lEqk7EDIsxMCHNCfJfytU4HTMo82ZCH6V9a8VSbQdsToeS"
    "d48lAFBdISCgniNLh5zJZiSnBcfcQImCfcWMwYTh8WT/DScuKIinTyaXpzxSa2OQV51tm5zKkl9/I392KZO34SQ+97sa7m6NJngH"
    "8L6LeWBoFXE23cxCqJMtcDxacOhvGfKLuLE5Ihifd12EeSx1CBazg3IKSpHWhuKbchN2KgArsicsQhS+XVu4QFULVVCSc4lDFjFo"
    "Ef4NjoiUUZYBHm6rVM2WcYEPQkCGY0jn2opU1Z7jjTWMWD99ErNWF7CoiULCZS88e8aJbrEDNy9hmoXMN328seMC/bp+9IhV6jEh"
    "SxBQsUPyaWBdbFtAXlATUP0ZFRts3HNrhpzdREHdIw0Vd8YTcb0lE+IJquB8ps1gUagdmG6Zh4Mo0B4FSJDInDjYGpyhChcIGTBV"
    "cFchaphMNNptHNBR2FhoCNNufPAXiHnFtkx7M0aRUMpULiYWux+j0UTkZzNAjCCndDXCz/irlzqYq49c1YBTKIK6eqg1G2O3Zkxe"
    "ypKXCAIomsgYOA0eUgVrzFCZ9xiA4348BwhOxCX/ZkSciGkYO5viz0/9c33g0HMxdEAbtfqJi9/UDPATgbL4s6DQjzE5Bzkg0Vca"
    "YXwe4TesdDnfU3hxNYB9zrraqlPjcMnpRb8kNAlxPoQnyt+WeJimj8QVKZNTEYV4ZQwPySyqnQCMYLtRPbElPyRZPdEh01ZKOvQr"
    "Ax+B/Aacxyy4ixWZ6naKVksiQwFLvqZ6YBQSUYA2PKlXJiCCiAUbE0ev+yIPcWxW0JfIJQkMaWa/wO6TECEktlzcgOlFVH3Jw2FL"
    "enAUPdCzNVs7MfblZRo6YO0n3dLE3DxCqfSz2Ohd9hlsXE7+ADiGa8DxCFARkjYjBUcMVD8S36M6SrmPC7Mt0jcIrLIIyRxehVSQ"
    "NsVYksadyvP9hjY2YDzIZYtAs6TVDaOKCvkEAHp2yRkUPL3oEPBLi/ovhD7xUk8EQ1pf6wIcE/i/dxLZstIxDSR56pbqa+1J9VjI"
    "lNSNZaiS2Oxd/T/6TAiFFocN+eBEHF4qxHw7uFOIdoyXCq8fpYiKWyIf6iUQEe8S4qrAfrmFi5aNrQ4gCNoP5wOb3TmCdvBD3JTO"
    "fpQIlgXpWOF261eyFkg4SM2pfaS8qdABkM2TC6hsmYmr786/pgTpuTCU1ImsZdepTS/CtQZ5KO5PXQOCOMkHAtnysVWgDdJ9wpH4"
    "t94AOIWWnCy70iwFIOcEfI7VSLp76Q4qtg3Zw1CaG3QoGT/87urNq9Ss9DB/z60otxIhF3KjCeaB0gQvKzZUTwx6FiojRN5Y57Io"
    "jZ4RW9uCRhawUGp99roekulRpLAqfosGCQJ9TLbgqhLuASYbVUpy6HfItdlVs8NmCY8cQ7vYs4d2/WiQMKkqYB3JXhQaakktVMOF"
    "EIa6vuUzPcij2cCcsb8RH9rJs9PFOPyh39NL+n1e8NDZM/4S5qcXPLTkz8swcQCswTOXQwEsczIdign3HHvtVXh8yk88HfdfzhQ/"
    "fknqp/0GqoalpxdBo5F4BYzuuiwgOFfAVaroq/5BR0otJteMqcXkeOrr/PsazRtty9ji7rWawy7zf7hn8xpV+0G0k5hU4YciKMsy"
    "8RJVUGkl3TnW7QKUxxrbRtLsH/1H4rJIUkAIVf4xPucmMnRoHNbtagUTLWUehwlcTvnwREVPjLtnw4pAllwLnp2eTk5HSSIn1VlI"
    "pnEPeGEOSqEelj4nASVdcXJqLbrThjlliKyKvhDnaw1ZY39KOODbFqlml2TQfWy4b0K7Ti0F++2rL6eThzHe9rk3SWzUL62GH0bi"
    "B6kdV9fOgy24QQqbDNSMCSWmdCrTwgDYgVLzV6k2iPHXc2EWy7FFG8oTvmMksMkbqUvuJgeJmnF14DcXefoPu03q8wWz/Dg9ms4M"
    "ho+MjkIOOvR3RM9xboCc4H+cksAYqXH8eARi+qVV/pjuJ6nYEXxvMOo3nceL873naCKbpIf/VnREp5x90ild2utz3p1ER6qFzMr9"
    "LyVvekuAFbYvppLAQCCB4vhqhDtqLoMGKQ4EX4Kk0s1X4psU+xkRQi/RwCzO7zd1OraH5Bbn22IX8RFo4s2dSJiJ71/88GdI4h6a"
    "CNCf27w5AFyJdFXedVBY/0nn0HuEO68OxpAeplelXczE8ePHANrPDtX+8eeAcejY71tft2zy6/RCpGkN6Y6ECDskTsb+1yPxFkvf"
    "vvwGxG+SgEO96mLJTqVXPVy0sNQRmzW6j6+bnc9KFe4X9gwgoiVnyNqKGr6spiULazdiR6dx4uPl5ATFp+Ab+C7hfXwWSysuQGSd"
    "4C74Qp2EjIBBRW8E6KUK1VYZjpnhAOG+jSoVgJXOdx+JRK+SJ4u/0a8s/65n/a0/HlDA3Jr5Ar7ZpM6mF3vo4W/hjYylUg+KEOHX"
    "VYs+cAkcyPwkLeOmJVV7xe/GClmWjU0Fwl+JleCxOSXJOLlv4TD/p02MNrvx87Yp+y71KFRGXDGdn3Jh9CQUdKFsC9OSh864uHo6"
    "mJ6ILxnCgyTVNTFIf3XfONE5197XbjYGm4/6k41ss+JqYPzw1eTh1fnDq1f84eLh5WkcefYkjbxMa76Zp09Xae4sjnz99M7qyOJw"
    "l93OgTOvGtObIBPHsO6HgU740mkVYYZUtrv30dCTfHk8GLJloYu9kXe1yrUsZ8MxNuZlX/SGqjUUytPzMHLngegJtv7ZZFhZs+ue"
    "9pVtmI1++pS4s4t+5/seOKyppzKJiJcsc8dvli3q0OOjapvF4YyCKK6kWwcCgqPqLiVUfjulaor4i2TgUu7m4Z6RriZGcWUuUY3f"
    "+7p4HKOEV/xu8ZRoNNS/WWEZoXEBKAQM6Fu6VfpH90ZvP0UjFa8tcn295hvG1ELxDZP2KCsgmN5OUqtaW6e9vlEhC9sy3S50JET5"
    "PLtxWWpW998uIbNwi0e1Jp+u4F0alSGNUC9ew1ZJFrE9WvIyaMV350Ztxb9+eE39a/gfBdysclXzGSpP5mydt9VfIZylLN2nGafn"
    "AXXL93yj3Fbj+CY0q9S9sSp+/Om+OOyG78KxvyT7PQCefQKAF/8HAL5/9ebt66v3r3BqWRRiB6ML1J9dkQjc0Xsvwt3XsdccVIfh"
    "zVK6rqL/QSDhs5O7zQTfdpCYdFt07AY96ZbuTBqkZYfC4wX1tklgYxfWO8qmDB5+MQ4MNi5e3OF8W3rrs7YGmrvD9QSrjGHlTpJI"
    "akv4Kgr9keXyP6pJmvD5Ua7W6HvjdSoA+eC3/wItVhmTiiMAAA=="
)

_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode('utf-8'))
for _rel, _content in _files.items():
    _path = REPO / _rel
    _path.parent.mkdir(parents=True, exist_ok=True)
    _path.write_text(_content, encoding='utf-8')
print(f'wrote {len(_files)} file(s)')
for _rel in sorted(_files):
    print('  ', _rel)


### Verify the modules import and the discriminator works

A quick sanity check on the property everything else depends on.


In [ ]:
from lma.data.language_id import IndicLanguageIdentifier
from lma.data.normalize import IndicNormalizer, get_profile
from lma.data.clean import DocumentCleaner, CleanerConfig

ident = IndicLanguageIdentifier()
hindi = ('भारत दक्षिण एशिया में स्थित एक विशाल देश है और इसकी राजधानी नई दिल्ली है। '
         'यहाँ हिन्दी भाषा बोली जाती है और गंगा नदी को पवित्र माना जाता है। '
         'उन्होंने कहा कि यह काम किया गया था और लोग इसे पसंद करते हैं।')
nepali = ('नेपाल दक्षिण एसियामा रहेको एउटा सानो देश हो र यसको राजधानी काठमाडौं हो। '
          'यहाँ नेपाली भाषा बोलिन्छ र सगरमाथा संसारकै अग्लो हिमाल हो। '
          'उनले भनेका छन् कि यो काम गरेको छ र धेरै मानिसहरू कृषिमा निर्भर छन्।')

for name, text in [('Hindi', hindi), ('Nepali', nepali)]:
    v = ident.identify(text)
    print(f'{name:7s} -> {v.language:8s} hi={v.hindi_score:.3f} ne={v.nepali_score:.3f}')

assert ident.accepts(nepali, 'ne'), 'Nepali not recognised'
assert not ident.accepts(hindi, 'ne'), 'Hindi would contaminate the Nepali corpus'
print('\nHindi/Nepali separation verified.')


## 4 · Scale the sources up

Widens the Sangraha shard pattern from one shard to all 25 and drops the
per-source document caps.

**The resource gap is visible right here:** Sangraha carries **25 verified
Nepali shards against 100 for Hindi** — a 4× difference in the largest open
corpus for each language, before any filtering. That gap is the phenomenon the
project studies, so report it rather than work around it.


In [ ]:
import yaml

manifest = REPO / 'nepali/configs/sources.yaml'
cfg = yaml.safe_load(manifest.read_text())
for src in cfg['sources']:
    if src['source_id'] == 'sangraha_ne':
        src['data_files'] = 'verified/nep/*.parquet'
        src.pop('limit', None)
    elif src['source_id'] == 'wikipedia_ne':
        src.pop('limit', None)
manifest.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

for s in cfg['sources']:
    if s.get('enabled', True):
        print(f"  {s['source_id']:30s} {s['collection']:10s} "
              f"{s.get('data_files') or s.get('config') or s.get('path', '')}")


## 5 · Build the corpus

**The long cell — 1 to 3 hours.**

Nepali is the lower-resource language, so this exhausts its sources rather than
reaching the 500M-token target. That is the expected result and the assignment
permits it explicitly: *"If the lower-resource language cannot reach 500M from
public sources, collect the maximum feasible amount, report the exact token
count, and justify the shortfall."*

### If the session ends first

Re-run **this same cell**. `--resume` reconciles the output files against the
checkpoint, restores the deduplication index, and skips what each source already
yielded. A resumed run produces a byte-identical corpus to an uninterrupted one
— verified by test, including after `kill -9`.

For an unattended run use **Save Version → Save & Run All (Commit)**; an
interactive session stops when you close the browser.


In [ ]:
def run(script, *extra):
    """Run a pipeline script, streaming output live rather than buffering it."""
    cmd = [sys.executable, str(REPO / 'scripts' / script), *extra]
    print(' '.join(cmd), '\n')
    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    return proc.wait()

rc = run('build_corpus.py',
         '--config',  str(REPO / 'nepali/configs/pipeline.yaml'),
         '--sources', str(REPO / 'nepali/configs/sources.yaml'),
         '--profile', 'full',
         '--target-tokens', '500000000',
         '--checkpoint-dir', str(REPO / 'checkpoints_ne'),
         '--checkpoint-every', '25000',
         '--resume',
         '--report-every', '20000')
print('\nexit code:', rc)


### Progress

Safe to run in a separate cell while the build is going, or in a new session to
see how far the previous one got.


In [ ]:
import json

state = REPO / 'checkpoints_ne' / 'ne_ingest_state.json'
if state.exists():
    st = json.load(open(state))
    print(f"segment     : {st['segments']}")
    print(f"docs seen   : {st['seen']:,}")
    print(f"docs kept   : {st['kept']:,}")
    print(f"train chars : {st['train_chars']:,}")
    print(f"elapsed     : {st['elapsed_seconds'] / 3600:.1f} h")
else:
    print('No checkpoint yet - the run has not saved one, or it finished and cleared it.')


## 6 · Train the tokenizer and count tokens

Trained from scratch on the Nepali training split only — no pretrained
tokenizer, as the assignment requires.

The vocabulary size is **chosen empirically, not asserted**: candidates are
trained across sizes and both algorithms, each measured on the same held-out
text, and the winner is the lowest fertility subject to an unknown-rate ceiling.
Near-ties resolve toward the smaller vocabulary, because embedding parameters
scale with vocabulary size and Phase 2 has a fixed ~25M-parameter budget.


In [ ]:
rc = run('train_tokenizer.py',
         '--config', str(REPO / 'nepali/configs/pipeline.yaml'),
         '--profile', 'full', '--sweep', '--use-sweep-best')
print('\nexit code:', rc)

# Exact token counts - the number the report must cite, measurable only once
# the tokenizer exists.
rc = run('tokenize_corpus.py', '--config', str(REPO / 'nepali/configs/pipeline.yaml'))
print('\nexit code:', rc)


## 7 · Results

Everything the Phase 1 report needs from the Nepali half.


In [ ]:
reports = REPO / 'nepali/reports'

tokens_file = reports / 'token_counts.json'
if tokens_file.exists():
    t = json.load(open(tokens_file))
    print('=== TOKEN COUNTS ===')
    print(f"  training tokens : {t['train_tokens']:,}")
    print(f"  all splits      : {t['total_tokens']:,}")
    print(f"  vs 500M target  : {100 * t['achieved_fraction_of_target']:.2f}%")
    if not t['target_met']:
        print(f"  shortfall       : {t['shortfall_tokens']:,} (report and justify this)")

stats_file = reports / 'corpus_stats.json'
if stats_file.exists():
    d = json.load(open(stats_file))
    print('\n=== CORPUS ===')
    print(f"  documents : {d['totals']['documents']:,}")
    print(f"  words     : {d['totals']['words']:,}")
    share = d['collection_split']['manual_word_fraction']
    verdict = 'OK' if share >= 0.20 else 'BELOW THE 20% REQUIREMENT'
    print(f"  manual    : {100 * share:.1f}%  [{verdict}]")
    print('\n  by source:')
    for src, words in sorted(d['by_source']['words'].items(), key=lambda kv: -kv[1]):
        print(f'    {src:32s} {words:>12,} words')

tok_file = reports / 'tokenizer_stats.json'
if tok_file.exists():
    m = json.load(open(tok_file))['metrics']
    print('\n=== TOKENIZER (held-out) ===')
    print(f"  algorithm       : {m['model_type']}")
    print(f"  vocabulary      : {m['vocab_size']:,}")
    print(f"  fertility       : {m['fertility']:.3f} tokens/word")
    print(f"  unknown rate    : {m['unk_rate']:.2e}")
    print(f"  byte-fallback   : {100 * m['byte_fallback_rate']:.4f}%")
    print(f"  chars per token : {m['chars_per_token']:.3f}")


### Corpus purity

The check that matters for this language pair: how much Hindi, if any, survived
into the Nepali corpus. Both are Devanagari, so this is the contamination path
a script check could never catch.


In [ ]:
from lma.data.splits import iter_jsonl

train = REPO / 'nepali/data/processed/ne_train.jsonl'
if train.exists():
    counts = {}
    for i, row in enumerate(iter_jsonl(train)):
        if i >= 3000:
            break
        lang = ident.identify(row['text']).language
        counts[lang] = counts.get(lang, 0) + 1
    total = sum(counts.values()) or 1
    print('verdicts on a 3,000-document sample:', counts)
    print(f"  identified as Nepali : {100 * counts.get('ne', 0) / total:5.1f}%")
    print(f"  Hindi contamination  : {100 * counts.get('hi', 0) / total:5.1f}%")
    print(f"  unknown              : {100 * counts.get('unknown', 0) / total:5.1f}%")
else:
    print('corpus not built yet')


## 8 · Download the results

The cell below gathers the small artifacts — tokenizer, vocabulary, statistics
— into one folder. Commit a version (**Save Version → Save & Run All**) and
they appear under the notebook's **Output** tab, ready to download and add to
your repository.

The corpus files themselves are large and belong on Drive or as a Kaggle
Dataset (Output tab → New Dataset), with the link in your README — that is what
the assignment asks for.


In [ ]:
OUT = REPO / 'nepali_artifacts'
OUT.mkdir(exist_ok=True)
for sub in ['tokenizer', 'reports', 'configs']:
    src = REPO / 'nepali' / sub
    if src.exists():
        shutil.copytree(src, OUT / sub, dirs_exist_ok=True)

total_kb = 0
for path in sorted(OUT.rglob('*')):
    if path.is_file():
        kb = path.stat().st_size / 1024
        total_kb += kb
        print(f'{kb:9.1f} KB  {path.relative_to(OUT)}')
print(f'{total_kb / 1024:9.1f} MB  TOTAL')

print('\nCorpus files (large - Drive or Kaggle Dataset, not git):')
proc = REPO / 'nepali/data/processed'
if proc.exists():
    for f in sorted(proc.glob('*.jsonl')):
        print(f'  {f.stat().st_size / 1e6:8.1f} MB  {f.name}')

print(f'\n/kaggle/working free: {shutil.disk_usage(REPO).free / 1e9:.1f} GB')
